In [2]:
import sys
sys.path.append('..')
import pandas as pd
from pathlib import Path
from src.core.data_preprocessing import LinearPreprocessor, TreePreprocessor
from src.core.data_splitter import DataSpliter

In [3]:
SPLITS_DIR = Path('../data/splits')
PROCESSED_DIR = Path('../data/processed')

In [6]:
splitter = DataSpliter(target_col='SUSCEP')
splitter.load(SPLITS_DIR)

train_df, val_df, test_df = splitter.get_splits()
train_val_df = splitter.get_train_val()

print(f"Train    : {len(train_df):,}")
print(f"Val      : {len(val_df):,}")
print(f"Test     : {len(test_df):,}")
print(f"Train+Val: {len(train_val_df):,}")

Total samples: 144401
Train samples: 101080 (70.00%)
Val samples: 21660 (15.00%)
Test samples: 21661 (15.00%)
Train    : 101,080
Val      : 21,660
Test     : 21,661
Train+Val: 122,740


In [8]:
train_df.columns = train_df.columns.str.strip()
val_df.columns = val_df.columns.str.strip()
test_df.columns = test_df.columns.str.strip()
train_val_df.columns = train_val_df.columns.str.strip()

In [9]:
print("\n{'='*50}")
print("PHASE 1 — Fit trên train set")
print('='*50)

# --- Tree ---
tree_p1 = TreePreprocessor()
p1_tree_train = tree_p1.fit_transform(train_df)
p1_tree_val   = tree_p1.transform(val_df)
p1_tree_test  = tree_p1.transform(test_df)

# --- Linear ---
linear_p1 = LinearPreprocessor()
p1_linear_train = linear_p1.fit_transform(train_df)
p1_linear_val   = linear_p1.transform(val_df)
p1_linear_test  = linear_p1.transform(test_df)


{'='*50}
PHASE 1 — Fit trên train set
[TreePreprocessor] fitting on 101,080 rows...
[TreePreprocessor] fit done ✅

[LinearPreprocessor] fitting on 101,080 rows...
  Skew transforms:
    'Slope': skew=-0.834 → λ=2.4231
    'TWI': skew=1.170 → λ=-1.5698
    'FA': skew=3.007 → λ=-0.4755
    'Rainfall': skew=0.579 → λ=-0.8657
  VIF analysis:
    Drop 'TWI' (VIF=20.42)
    Features giữ lại: ['Slope', 'Curvature', 'Aspect_sin', 'Aspect_cos', 'FA', 'Drainage', 'Rainfall']
[LinearPreprocessor] fit done ✅


In [10]:
print("\n{'='*50}")
print("PHASE 2 — Fit trên train+val set")
print('='*50)

# --- Tree ---
tree_p2 = TreePreprocessor()
p2_tree_train_val = tree_p2.fit_transform(train_val_df)
p2_tree_test      = tree_p2.transform(test_df)

# --- Linear ---
linear_p2 = LinearPreprocessor()
p2_linear_train_val = linear_p2.fit_transform(train_val_df)
p2_linear_test      = linear_p2.transform(test_df)


{'='*50}
PHASE 2 — Fit trên train+val set
[TreePreprocessor] fitting on 122,740 rows...
[TreePreprocessor] fit done ✅

[LinearPreprocessor] fitting on 122,740 rows...
  Skew transforms:
    'Slope': skew=-0.832 → λ=2.4143
    'TWI': skew=1.173 → λ=-1.5667
    'FA': skew=3.009 → λ=-0.4756
    'Rainfall': skew=0.577 → λ=-0.8616
  VIF analysis:
    Drop 'TWI' (VIF=20.40)
    Features giữ lại: ['Slope', 'Curvature', 'Aspect_sin', 'Aspect_cos', 'FA', 'Drainage', 'Rainfall']
[LinearPreprocessor] fit done ✅


In [12]:
def save_csv(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"  Saved: {path} ({len(df):,} rows)")



In [13]:
print("\nSaving...")
# Phase 1
save_csv(p1_tree_train,    PROCESSED_DIR / 'phase1/tree/train.csv')
save_csv(p1_tree_val,      PROCESSED_DIR / 'phase1/tree/val.csv')
save_csv(p1_tree_test,     PROCESSED_DIR / 'phase1/tree/test.csv')

save_csv(p1_linear_train,  PROCESSED_DIR / 'phase1/linear/train.csv')
save_csv(p1_linear_val,    PROCESSED_DIR / 'phase1/linear/val.csv')
save_csv(p1_linear_test,   PROCESSED_DIR / 'phase1/linear/test.csv')

# Phase 2
save_csv(p2_tree_train_val,   PROCESSED_DIR / 'phase2/tree/train_val.csv')
save_csv(p2_tree_test,        PROCESSED_DIR / 'phase2/tree/test.csv')

save_csv(p2_linear_train_val, PROCESSED_DIR / 'phase2/linear/train_val.csv')
save_csv(p2_linear_test,      PROCESSED_DIR / 'phase2/linear/test.csv')

print("\nDone! Cấu trúc đã tạo:")
for p in sorted(PROCESSED_DIR.rglob('*.csv')):
    print(f"  {p}")



Saving...
  Saved: ..\data\processed\phase1\tree\train.csv (101,080 rows)
  Saved: ..\data\processed\phase1\tree\val.csv (21,660 rows)
  Saved: ..\data\processed\phase1\tree\test.csv (21,661 rows)
  Saved: ..\data\processed\phase1\linear\train.csv (101,080 rows)
  Saved: ..\data\processed\phase1\linear\val.csv (21,660 rows)
  Saved: ..\data\processed\phase1\linear\test.csv (21,661 rows)
  Saved: ..\data\processed\phase2\tree\train_val.csv (122,740 rows)
  Saved: ..\data\processed\phase2\tree\test.csv (21,661 rows)
  Saved: ..\data\processed\phase2\linear\train_val.csv (122,740 rows)
  Saved: ..\data\processed\phase2\linear\test.csv (21,661 rows)

Done! Cấu trúc đã tạo:
  ..\data\processed\phase1\linear\test.csv
  ..\data\processed\phase1\linear\train.csv
  ..\data\processed\phase1\linear\val.csv
  ..\data\processed\phase1\tree\test.csv
  ..\data\processed\phase1\tree\train.csv
  ..\data\processed\phase1\tree\val.csv
  ..\data\processed\phase2\linear\test.csv
  ..\data\processed\phase2